# SAM3 Roof Segmentation — Google Colab

This notebook runs SAM3 (Segment Anything 3) on satellite tiles to detect roofs.

**Why Colab?** SAM3 requires NVIDIA CUDA 12.6+. If your local machine has an AMD GPU,
run this notebook on Colab's free T4 GPU.

## Workflow
1. Upload your satellite tiles (from `data/raw/tiles/{suburb}/`)
2. This notebook runs SAM3 on each tile with the text prompt "roof"
3. Download the resulting masks and metadata back to `data/processed/masks/{suburb}/`

## Prerequisites
- HuggingFace account with access to SAM3 model weights
- Satellite tiles already downloaded locally via `run_stage1.py`

In [ ]:
# Step 1: Install SAM3 and dependencies
!pip install -q torch torchvision
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip install -e .
%cd ..

In [ ]:
# Step 2: Authenticate with HuggingFace (for model weights)
from huggingface_hub import login
login()  # Paste your HF token when prompted

In [ ]:
# Step 3: Upload your tiles
# Option A: Upload a zip file
from google.colab import files
import zipfile
import os

print("Upload your tiles zip file (e.g., richmond_tiles.zip):")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall('tiles')
        print(f"Extracted {filename} to tiles/")

# Option B: Mount Google Drive (uncomment below)
# from google.colab import drive
# drive.mount('/content/drive')
# TILES_PATH = '/content/drive/MyDrive/raising_rooves/tiles'

In [ ]:
# Step 4: Load SAM3 model
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3 model...")
model = build_sam3_image_model()
processor = Sam3Processor(model)
print("Model loaded!")

In [ ]:
# Step 5: Run segmentation on all tiles
import json
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm

TILES_DIR = Path('tiles')  # Adjust if using Google Drive
OUTPUT_DIR = Path('masks')
OUTPUT_DIR.mkdir(exist_ok=True)

tile_paths = sorted(TILES_DIR.glob('**/*.png'))
print(f"Found {len(tile_paths)} tiles to process.")

for tile_path in tqdm(tile_paths, desc="Segmenting roofs"):
    try:
        image = Image.open(tile_path).convert('RGB')
        inference_state = processor.set_image(image)
        output = processor.set_text_prompt(state=inference_state, prompt='roof')

        masks = output['masks'].cpu().numpy()
        boxes = output['boxes'].cpu().numpy()
        scores = output['scores'].cpu().numpy()

        # Build combined binary mask
        h, w = masks.shape[1], masks.shape[2]
        combined_mask = np.zeros((h, w), dtype=bool)
        segments = []

        for i in range(len(scores)):
            mask_i = masks[i] > 0.5
            pixel_count = int(mask_i.sum())
            if pixel_count < 50:
                continue

            combined_mask |= mask_i
            ys, xs = np.where(mask_i)

            segments.append({
                'id': i,
                'pixel_count': pixel_count,
                'bbox': [int(v) for v in boxes[i]],
                'centroid': [float(xs.mean()), float(ys.mean())],
                'confidence': float(scores[i]),
            })

        # Save mask
        stem = tile_path.stem
        mask_img = Image.fromarray((combined_mask * 255).astype(np.uint8), mode='L')
        mask_img.save(OUTPUT_DIR / f'{stem}_mask.png')

        # Save metadata
        metadata = {
            'tile': tile_path.name,
            'total_roof_pixels': int(combined_mask.sum()),
            'num_segments': len(segments),
            'segments': segments,
        }
        (OUTPUT_DIR / f'{stem}_meta.json').write_text(json.dumps(metadata, indent=2))

    except Exception as e:
        print(f"Error processing {tile_path.name}: {e}")

print(f"\nDone! {len(list(OUTPUT_DIR.glob('*_mask.png')))} masks generated.")

In [ ]:
# Step 6: Preview a result (optional)
import matplotlib.pyplot as plt

# Pick a random tile to visualise
sample_mask = sorted(OUTPUT_DIR.glob('*_mask.png'))[0]
sample_tile = TILES_DIR / sample_mask.name.replace('_mask.png', '.png')

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

if sample_tile.exists():
    axes[0].imshow(Image.open(sample_tile))
    axes[0].set_title('Original Tile')
    axes[0].axis('off')

axes[1].imshow(Image.open(sample_mask), cmap='gray')
axes[1].set_title('Roof Mask')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Download results
# Option A: Download as zip
import shutil
shutil.make_archive('masks_output', 'zip', 'masks')
files.download('masks_output.zip')

# Option B: Save to Google Drive (uncomment below)
# import shutil
# shutil.copytree('masks', '/content/drive/MyDrive/raising_rooves/masks', dirs_exist_ok=True)
# print('Saved to Google Drive!')